<h2><b>计算机高等教育通用教材</b></h2>
<h2>机器学习 Machine learning</h2>
<hr>
<h5>第一部分：监督学习 supervised learning</h5>
<h5>第六章：分类与边界寻找 支持向量机 SVM</h5>
<hr>
<h3><b>实验六：基于 SVM 预测乳腺癌良恶性与参数寻优</b></h3>
<hr>
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html'>查看SVC源代码(sklearn)</a><br>
<br>

> **适合人群** ：你已经见识过了依靠概率猜谜的“朴素贝叶斯”，也体验了靠问问题分叉的“决策树”。今天，我们将接触分类算法中极具几何美感的霸主——**支持向量机（Support Vector Machine, SVM）**。
> 别被这个 "向量机" 名字吓到。学完本章你会发现，SVM 的本质，仅仅是在两种数据之间的边界上做分割
<hr>

#### 第0步：测试python与虚拟环境

In [ ]:
print("Hello Support Vector Machine!")
import pip
print("Pip version:", pip.__version__)

<hr><hr>

#### 第一步：import库 & 导入数据
<hr>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_curve, auc, confusion_matrix

In [ ]:
# 教材中使用的是 breast-cancer-kaggle.csv。为了保证代码的一键可运行性，
# 我们直接调用 sklearn 内置的威斯康星州乳腺癌数据集（数据完全一致）。
# 该数据集包含 569 个肿瘤样本，30 个生理特征（如细胞核半径、纹理、周长、平滑度等）。
# 标签：0 代表恶性 (Malignant)，1 代表良性 (Benign)。

cancer_data = load_breast_cancer()
X_raw = cancer_data.data
y = cancer_data.target
feature_names = cancer_data.feature_names

# 为了方便人类观察，装进 DataFrame
df = pd.DataFrame(X_raw, columns=feature_names)
df['诊断结果'] = y
# 将 0 和 1 映射为汉字
df['诊断结果_汉字'] = df['诊断结果'].map({0: '恶性', 1: '良性'})

print(f'数据集大小: {df.shape}')
df.head()

<hr><hr>

#### 第二步：查看数据的基本信息与探查
<hr>

In [ ]:
df.info()
# 30 个 float64 类型的连续特征，没有任何空值。非常纯净的数据集。

In [ ]:
# 看看良性和恶性肿瘤的比例
label_count = df['诊断结果_汉字'].value_counts()
print("肿瘤样本分布：\n", label_count)

# 良性有 357 个，恶性有 212 个。
# 虽然良性稍微多一点，但整体分布还算平衡。如果恶性只有 5 个，良性有 500 个，那就叫“样本严重不平衡”，
# 处理那种情况就需要用到后续讲的 AUC 曲线了。

<hr><hr>

#### 第三步：数据转换与特征降维 (PCA)
<hr>

30 个特征太多了！这不仅会让模型训练变慢，还会包含很多相似的“废话特征”。
比如“平均半径”和“平均周长”，这两个特征几乎表达的是同一个意思（圆越大，周长肯定越长）。
留下这么多冗余特征，模型容易钻牛角尖（过拟合）。

怎么办？我们要祭出数据分析领域的“降维打击”法宝 —— **主成分分析（PCA）**。

In [ ]:
# 1. 拆分特征和标签
X = df.drop(columns=['诊断结果', '诊断结果_汉字']).values
y = df['诊断结果'].values

# 2. 切分考卷（80%训练，20%测试）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    random_state=42, 
    test_size=0.2
)

# 3. 数据标准化 (StandardScaler)
# SVM 对数据的绝对大小极其敏感！因为它是在空间里算距离的算法。
# 如果“周长”是 120，“平滑度”是 0.01，那距离计算会被“周长”彻底带偏。
# 所以，在用 SVM 和 PCA 之前，必须做标准化！
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

# 4. 特征降维 (PCA)
# 我们要求 PCA 把 30 个特征，浓缩成 2 个最核心的“超级特征”，这样我们甚至能在二维平面上把它们画出来！
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"降维前，训练集特征维度: {X_train_scaled.shape}")
print(f"降维后，训练集特征维度: {X_train_pca.shape}")
# 从 30 列变成了 2 列！

In [ ]:
# 配置中文字体
import matplotlib.font_manager as fm
zh_fonts = [f.name for f in fm.fontManager.ttflist 
            if any(kw in f.name for kw in ['Hei', 'Song', 'CJK', 'Chinese', 'SC', 'TC', 'Gothic', 'SimHei'])]
if zh_fonts:
    plt.rcParams['font.family'] = zh_fonts[0]
plt.rcParams['axes.unicode_minus'] = False 

# 让我们看看降维成 2 个特征后，数据长什么样
plt.figure(figsize=(8, 6))
# 画恶性肿瘤（标签 0）
plt.scatter(X_train_pca[y_train == 0, 0], X_train_pca[y_train == 0, 1], color='red', label='恶性', alpha=0.6)
# 画良性肿瘤（标签 1）
plt.scatter(X_train_pca[y_train == 1, 0], X_train_pca[y_train == 1, 1], color='green', label='良性', alpha=0.6)

plt.title('PCA降维后的乳腺癌数据分布 (30维压缩至2维)')
plt.xlabel('超级特征 1 (第一主成分)')
plt.ylabel('超级特征 2 (第二主成分)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 结果分析：
# 你看！哪怕我们把 30 个指标暴力压缩成了 2 个，红色点和绿色点依然在空间中泾渭分明。
# 既然它们分得这么开，接下来就该 SVM 出场，在这两堆点中间“修一堵墙”了！

<hr><hr>

#### 第四步：模型训练与自动化参数寻优 (GridSearchCV)
<hr>

SVM 虽然强大，但它也是最吃参数配置的模型。它有几个致命的超参数：
- `C` (惩罚系数)：C 越大，墙修得越死板（不允许错误）；C 越小，墙修得越宽容。
- `kernel` (核函数)：是用直线的墙（linear），还是扭曲的墙（rbf）？

如果你一个一个手动去试，那能把人累死。
工业界标准做法：用 **GridSearchCV（网格搜索交叉验证）** 帮我们全自动把所有参数组合试一遍，并选出最高分！

In [ ]:
print("开始全自动参数寻优，请稍候...")

# 1. 定义我们想试的参数组合菜单
param_grid = {
    'C': [0.1, 1, 10, 100],               # 试 4 种惩罚力度
    'kernel': ['linear', 'rbf'],          # 试 2 种墙的形状
    'gamma': ['scale', 'auto', 0.1, 0.01] # 核函数的细微调整
}

# 2. 把基础模型和菜单交给 GridSearchCV，它会自动做 5 折交叉验证（cv=5）
grid_search = GridSearchCV(
    SVC(probability=True), # probability=True 是为了后面画 ROC 曲线
    param_grid, 
    cv=5, 
    scoring='accuracy',
    n_jobs=-1 # 满功率运行你的电脑 CPU
)

# 3. 让它在训练集上自己去试
grid_search.fit(X_train_pca, y_train)

# 4. 考试结束，交卷！
best_model = grid_search.best_estimator_
print(f"\n寻优完毕！GridSearchCV 找到的最优参数是：\n{grid_search.best_params_}")
print(f"在 5 折交叉验证中的平均最高分：{(grid_search.best_score_ * 100):.2f}%")

<hr><hr>

#### 第五步：错分样本与 ROC 曲线分析（突破准确率的谎言）
<hr>

把模型用到未知的期末考试卷（测试集）上。

In [ ]:
# 用自动挑选出的最好模型进行预测
y_pred = best_model.predict(X_test_pca)
acc = accuracy_score(y_test, y_pred)
print(f"最终最优 SVM 模型在测试集上的准确率: {(acc * 100):.2f}%")

##### 为什么我们还需要 ROC 和 AUC？

假设有一个极其庸医的模型，它的规则是：“只要病人来，我都告诉他没事（良性）。”
如果今天来了 100 个病人，99 个是真没病，1 个是绝症。
这个庸医猜对了 99 次，准确率（Accuracy）高达 99%！
你觉得这是个好模型吗？不，那 1 个被误诊的绝症病人被耽误了治疗，这是**致命的假阴性 (False Negative)**。

在医疗、金融反欺诈这种领域，**光看准确率是骗人的**。
宁可误报（把健康人吓一跳去复查），也绝对不能漏报（把癌症病人放跑）。

怎么评估这种权衡能力？我们需要 **ROC 曲线** 和它下方的面积 **AUC**。
- 横轴 (FPR)：假阳率。健康人被误诊为癌症的比例（受点惊吓，可接受）。
- 纵轴 (TPR)：真阳率。真正的癌症病人被成功查出来的比例（必须越高越好）。

In [ ]:
# 1. 获取模型对每个测试样本的置信度概率（而不是直接给出 0 或 1 的死结论）
# 注意：在 sklearn 中，因为恶性是 0，良性是 1，为了符合医学逻辑（把恶性当做阳性 Positive 事件），
# 我们需要预测它是恶性(0)的概率。
y_prob_malignant = best_model.predict_proba(X_test_pca)[:, 0]

# 同样，要把真实的恶性标签翻转成 1，良性翻转成 0，让 roc_curve 函数能正确理解谁是正例
y_test_flipped = np.where(y_test == 0, 1, 0)

# 2. 计算 ROC 曲线的横纵坐标点
fpr, tpr, thresholds = roc_curve(y_test_flipped, y_prob_malignant)

# 3. 计算 AUC (ROC 曲线下的面积)
roc_auc = auc(fpr, tpr)

# 4. 画图
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC 曲线 (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='瞎蒙基准线')
plt.xlim([-0.02, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('假阳率 (FPR) - 误诊健康人的比例')
plt.ylabel('真阳率 (TPR) - 成功揪出癌症病人的比例')
plt.title('受试者工作特征曲线 (ROC)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

# 结论分析：
# AUC 的满分是 1.0。AUC 越接近 1.0，说明模型越能完美地把病人和健康人分开。
# 我们的模型 AUC 达到了 0.98+，这是一条极度贴近左上角的完美曲线，证明 SVM 模型对乳腺癌的判断能力极其出色！

<hr><hr>

#### 第六步：【拓展提高】剥开面纱看本质：SVM 究竟在干嘛？
<hr>

哪怕代码跑通了，很多人依然不知道 SVM 这个缩写到底是什么意思。
接下来，我用最通俗的语言，带你理解 SVM 的三大灵魂概念：**超平面、支持向量、核技巧**。

---

##### 1. 超平面 (Hyperplane) 与 间隔 (Margin)

想象一下，桌子上散落着一堆红豆（恶性肿瘤）和一堆绿豆（良性肿瘤）。
你的任务是拿一根笔直的木棍，把红豆和绿豆分开。
你可以斜着放，可以横着放，只要能分开就行。这根木棍，就叫**超平面**。

问题来了，哪种放法是最好的？

如果你把木棍紧紧贴着红豆放，那明天稍微有一颗新的红豆长得偏了一点，就越界被当成绿豆了。

SVM 的绝对原则是：**这根木棍，必须放在离两边最近的豆子都“尽可能远”的地方。**
木棍到最近的红豆的距离，加上木棍到最近的绿豆的距离，这个总宽度，被称为**间隔 (Margin)**。
SVM 寻找的，就是能够让这根木棍的“左右护城河（间隔）”达到最宽的那条线。

---

##### 2. 为什么叫“支持向量 (Support Vector)”？

在红豆和绿豆的海洋里，真的所有豆子都在决定那根木棍的位置吗？

根本不是！
那些远离边界、躲在后方的大量豆子，对木棍放哪毫无影响。你哪怕把它们全扔了，木棍依然在那。
真正决定木棍位置的，只有最靠近边界的那几颗极为凶险的、差一点就越界的豆子。

这几个极其关键的、在边界线边缘疯狂试探的数据点（或者说向量），就像支撑起这片隔离带的桥墩。
因此，它们被数学家们尊称为：**支持向量 (Support Vector)**。

SVM 的牛逼之处就在于此：不管你有几百万条数据，它最终只靠那几个最关键的“支持向量”来做决定，计算极其高效，这就是文档中说的**稀疏性**。

---

##### 3. 软间隔 (Soft Margin) 与 惩罚系数 C

现实世界没有那么完美。如果有一颗红豆，就是极其调皮地滚到了绿豆堆的最深处。
这时候如果你强迫症发作，非要找一根笔直的木棍把它们 100% 完美分开，这是绝对不可能的。

- 如果你硬要分开，你只能把木棍折弯成麻花（这叫**过拟合**）。
- SVM 提供了另一种选择：**软间隔**。它说：“算了，我允许这颗调皮的红豆站错地方，只要大部队分开就行。”

我们刚才在 GridSearchCV 里调的那个参数 **`C`**，就是你的强迫症指数。
- `C` 极其巨大：你是个严厉的暴君，绝对不允许哪怕一颗豆子站错位置，硬间隔。容易过拟合。
- `C` 非常小：你极其宽容，站错几颗也无所谓，只要大边界是对的就行。容易得到更稳健的模型。

---

##### 4. 终极魔法：核技巧 (Kernel Trick)

碰到最极端的情况：桌子上的红豆在中间围成了一个实心的圆圈，绿豆在外面围成了一个大圆圈。
你拿一根笔直的木棍，怎么划都没法把它们分开。这叫**线性不可分**。

SVM 该认输了吗？不。SVM 祭出了被誉为机器学习史上最震撼的数学魔法：**核技巧 (Kernel Trick)**。

既然在二维桌面（低维空间）用直线分不开，SVM 选择“猛拍一下桌子”！
所有的豆子都被震飞到了半空中（三维空间）。因为红豆在中间，绿豆在外面，它们在空中飞起的高度不一样。
在豆子腾空的一瞬间，SVM 掏出一张平整的硬纸板（超平面），“唰”地一下从红豆和绿豆之间水平插了过去！

在三维空间里，一张平坦的纸板轻松地把它们分开了。等豆子落回桌面，你再看那条切痕，桌面上的切痕变成了一个完美的圆形。

刚才你在代码里看到的参数 **`kernel='rbf'` (高斯核函数)**，就是这个帮数据“起飞腾空，升维打击”的魔法口诀。它能把原本在低维空间里扭曲纠缠的数据，映射到无限维的空间里，然后用最简单的一刀，切出最复杂的边界。

#### 总结

| 概念 | 大白话解释 | 实战注意事项 |
|------|------|------|
| **标准化 (StandardScaler)** | 统一单位尺度。 | SVM 在空间里算距离，如果不做标准化，模型会被大数字特征彻底毁掉。**必做**。 |
| **PCA (降维)** | 去掉冗余废话，把 30 维浓缩成 2 维。 | 防止过拟合并加速训练，同时能让我们把数据画在屏幕上。 |
| **GridSearchCV** | 穷举所有参数组合，全自动阅卷找最高分。 | SVM 对 `C` 和 `kernel` 极其敏感，绝不要相信直觉，一定要让机器自己搜最优解。 |
| **ROC / AUC** | 评估模型在“宁可错杀也不放过”时的权衡能力。 | 在医疗、反欺诈这种样本严重不平衡、且假阴性后果极度严重的场景，**AUC 远比 Accuracy 准确率重要**。 |
| **核技巧 (Kernel)** | 低维分不开？拍桌子让它升维。 | 遇到复杂非线性数据，果断选择 `kernel='rbf'`。 |

<br>

> **关键点**：这节课，我们用降维打击（PCA）、自动化武库（GridSearchCV）和顶级评估工具（ROC），把你武装到了牙齿。SVM 的那句核心哲学请牢记在心：**绝不要试图记住所有的数据点，只关注边界上最危险的那几个“支持向量”就足够了。**

<br>

<hr><hr>

## 实验六完成
<hr>

##### 此实验教材最近更新时间 2026年3月17日
<hr><hr>